## GymSense: Multimodal Workout Recommendation Agent

## Project Overview

GymSense is a multimodal workout recommendation agent developed using Python, LangChain, LangGraph, Gemini API, and Tavily. It accepts gym equipment information through either images or text descriptions and recommends suitable workouts based on available equipment and target muscle groups.

## Architecture and Methodology

<b>1) Multimodal Input Processing:</b> Accepts gym photographs (Base64 encoded) or text descriptions of available equipment.

<b>2) Tool-Calling Agent:</b> Uses an agent-based architecture to process user requests, determine available equipment, and generate relevant workout recommendations.

<b>3) Workout Recommendation:</b> Suggests suitable exercises based on the available gym equipment and the user's target muscle group.

<b>4) Tavily Web Search Integration:</b> Retrieves relevant web information to provide grounded workout recommendations.

<b>5) Conversational Memory:</b> Maintains conversation context to support follow-up questions and provide additional explanations about recommended exercises.

## Key Feature

The system supports both image and text-based inputs, allowing users to obtain workout recommendations based on their gym's available equipment, with conversational follow-ups for additional exercise guidance.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
tavily_client = TavilyClient()

@tool
def web_search(query:str) -> Dict[str, Any]:
    '''Search the web for information'''
    return tavily_client.search(query)

In [3]:
system_prompt='''
You are a personal gym trainer. The user will give you either a photo or a text
message describing the equipment available in their gymnasium.
If a photo is given, first list the equipment you can clearly identify in it.
Do not invent equipment that is not visible.
Using the web search tool, search the web for workouts that can be done with
that equipment. Return workout suggestions grouped by muscle group.
Only explain how to perform a workout (steps, sets/reps, form tips) if the
user requests it.
'''

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    tools = [web_search],
    system_prompt = system_prompt,
    checkpointer = InMemorySaver()
)

In [5]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id" : "1"}}
response = agent.invoke(
    {"messages":[HumanMessage(content="My gym has dumbbells, a flat bench and a pull-up bar. What workouts can I do?")]},
    config
)
#print(response['messages'][-1].content)
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='My gym has dumbbells, a flat bench and a pull-up bar. What workouts can I do?', additional_kwargs={}, response_metadata={}, id='60f0a505-5cc7-47f4-9347-ade90145f64b'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "dumbbell flat bench pull up bar workout muscle groups"}'}, '__gemini_function_call_thought_signatures__': {'call_8695771789357629077': 'EokBCoYBAWkUfRNMEztOthdza3tTdblddxS6d8qQxGEPrK6sOlNmMLoDey40X3GlDwTyYn/GcYmNAzllmNxAJ07LDR3cZzf97lnbW6u+aWYAnGhjekf05EkGLKQ5PEzHbBCF7ThYhkxsvUXte6k2hIq6ZPrWS9gMONAJdp2Q+QTZ4dS10CBZjEFGzt0='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1ee73-53f6-758b-b734-ea5c5ff47e27-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'dumbbell flat bench pull up bar workout muscle groups'}, 'id': 'call_8695771789357629077', 'type': '

In [6]:
#Image upload
from ipywidgets import FileUpload
from IPython.display import display
uploader = FileUpload(accept='.png, .jpg, .jpeg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png, .jpg, .jpeg', description='Upload')

In [7]:
import base64
uploaded_file = uploader.value[0]
content_mv = uploaded_file["content"]
mime_type = uploaded_file["type"]
img_bytes = bytes(content_mv)
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [8]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id" : "1"}}
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "This is my gym. What workouts can I do with this equipment?"},
    {"type": "image", "base64": img_b64, "mime_type": mime_type}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].text)

Based on the photo provided, here is the equipment I can clearly identify:
* **Two Treadmills** (cardio equipment)
* **One Elliptical Trainer** (cardio equipment)

*(Note: There may be more equipment out of frame, but based strictly on what is visible here, your options are focused entirely on cardiovascular conditioning and lower-body endurance.)*

Here are the workouts you can do grouped by muscle group/fitness goal using this cardio equipment:

### 1. Cardiovascular Endurance & Heart Health (Full Body)
* **Steady-State Running or Jogging (Treadmill):** Maintain a comfortable, consistent pace for 20 to 45 minutes to build aerobic endurance.
* **Incline Treadmill Walking:** Walk at a brisk pace on a high incline (e.g., 8–15% grade) to target your heart rate while putting less impact on your joints.
* **Elliptical Steady-State:** Maintain a steady rhythm pushing and pulling the handles while pedaling forward or backward.

### 2. Lower Body (Glutes, Hamstrings, Quadriceps, Calves)
* **H

In [9]:
#Follow up question to evaluate agent's memory
response = agent.invoke(
    {"messages": [HumanMessage(content="How to do rope cable curls? tell in two sentences.")]},
    config
)

print(response['messages'][-1].text)

Attach a rope handle to a low cable pulley, grab the ends with your palms facing each other, and keep your elbows pinned to your sides. Squeeze your biceps to curl the rope upward, slightly pulling your hands apart at the top before slowly lowering back down.
